Importing Libraries

In [44]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import layers

Loading Dataset

In [38]:
df = pd.read_csv('telco_customer_churn.csv')

In [39]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,Age,Region,tenure,PhoneService,MultipleLines,...,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,NumberOfReferrals,AvgMonthlyGBDownload,SupportCallsLast6Months,SatisfactionScore,Churn
0,8046-QZRTY,Female,0,Yes,Yes,46.0,Central,11,Yes,No,...,Month-to-month,No,Mailed check,22.54,203.23,1,0.0,2,1.0,Yes
1,7013-HTJAE,Male,0,No,No,60.0,West,72,Yes,Yes,...,One year,Yes,Electronic check,106.83,7702.50,2,6.0,1,5.0,No
2,5195-MNKHW,Female,0,No,Yes,66.0,South,10,Yes,No,...,Month-to-month,Yes,Mailed check,79.48,752.93,0,41.0,0,3.0,Yes
3,4258-TNLUK,Male,0,Yes,No,26.0,Central,19,Yes,No,...,Month-to-month,No,Mailed check,84.40,1572.13,0,11.0,2,4.0,No
4,2460-WBVMA,Female,1,Yes,Yes,61.0,North,37,No,No phone service,...,One year,No,Electronic check,94.59,3513.85,1,38.0,2,4.0,No


In [40]:
print(df.isnull().sum())

customerID                   0
gender                       0
SeniorCitizen                0
Partner                      0
Dependents                   0
Age                        201
Region                       0
tenure                       0
PhoneService                 0
MultipleLines                0
InternetService              0
OnlineSecurity               0
OnlineBackup                 0
DeviceProtection             0
TechSupport                  0
StreamingTV                  0
StreamingMovies              0
Contract                     0
PaperlessBilling             0
PaymentMethod                0
MonthlyCharges               0
TotalCharges               101
NumberOfReferrals            0
AvgMonthlyGBDownload        80
SupportCallsLast6Months      0
SatisfactionScore          122
Churn                        0
dtype: int64


Drops the customerID column.   
Handles blank spaces in TotalCharges and drops missing values (NaN).   
Converts the target column Churn into binary values (Yes $\rightarrow 1$, No $\rightarrow 0$).   
Applies One-Hot Encoding to all categorical feature columns.

In [41]:
df = df.copy()

# Drop customerID
df.drop(columns=["customerID"], inplace=True, errors="ignore")

# Remove extra spaces from text columns
for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].str.strip()

# Convert TotalCharges to numeric if the column exists
# Blank strings become NaN
if "TotalCharges" in df.columns:
    df["TotalCharges"] = pd.to_numeric(
        df["TotalCharges"], errors="coerce"
    )

# Convert Churn: Yes = 1, No = 0
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# Remove rows containing missing values
df.dropna(inplace=True)

# One-hot encode all remaining categorical feature columns
X = pd.get_dummies(
    df.drop(columns=["Churn"]),
    drop_first=False,
    dtype=int
)

# Target variable
y = df["Churn"].astype(int)

# Ensure all features are numeric
X = X.astype("float32")

print("Cleaned dataset shape:", df.shape)
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nFirst 5 rows of encoded features:")
display(X.head())


Cleaned dataset shape: (9530, 26)
Feature matrix shape: (9530, 55)
Target shape: (9530,)

Target distribution:
Churn
0    6993
1    2537
Name: count, dtype: int64

First 5 rows of encoded features:


,SeniorCitizen,Age,tenure,MonthlyCharges,TotalCharges,NumberOfReferrals,AvgMonthlyGBDownload,SupportCallsLast6Months,SatisfactionScore,gender_Female,...,StreamingMovies_Yes,Contract_Month-to-month,Contract_One year,Contract_Two year,PaperlessBilling_No,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0.0,46.0,11.0,22.540001,203.229996,1.0,0.0,2.0,1.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1,0.0,60.0,72.0,106.830002,7702.500000,2.0,6.0,1.0,5.0,0.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
2,0.0,66.0,10.0,79.480003,752.929993,0.0,41.0,0.0,3.0,1.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
3,0.0,26.0,19.0,84.400002,1572.130005,0.0,11.0,2.0,4.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
4,1.0,61.0,37.0,94.589996,3513.850098,1.0,38.0,2.0,4.0,1.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


Train-Test Split

In [42]:
RANDOM_STATE = 123

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,

)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])
print("Number of input features:", X_train.shape[1])

Training samples: 7624
Testing samples: 1906
Number of input features: 55


Building a Neural Network Model

In [45]:
def build_model(input_dim):
    model = keras.Sequential([
        keras.Input(shape=(input_dim,)),
        layers.Dense(8, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

# Build the first model
tf.keras.utils.set_random_seed(42)

model_unscaled = build_model(X_train.shape[1])

# Display model architecture
model_unscaled.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_18 (Dense)                │ (None, 8)              │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 457 (1.79 KB)

 Trainable params: 457 (1.79 KB)

 Non-trainable params: 0 (0.00 B)

Training Keras Model Without Feature Scaling

In [46]:
# Train the model without feature scaling
history_unscaled = model_unscaled.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    verbose=1
)

# Evaluate on the test set
loss_unscaled, accuracy_unscaled = model_unscaled.evaluate(
    X_test,
    y_test,
    verbose=0
)

print(f"Test Loss without Scaling: {loss_unscaled:.4f}")
print(f"Test Accuracy without Scaling: {accuracy_unscaled * 100:.2f}%")

Epoch 1/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4685 - loss: 37.9050
Epoch 2/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6486 - loss: 1.0473
Epoch 3/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6625 - loss: 0.7962
Epoch 4/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6882 - loss: 0.6679
Epoch 5/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7100 - loss: 0.6047
Epoch 6/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7251 - loss: 0.5721
Epoch 7/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7320 - loss: 0.5541
Epoch 8/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7356 - loss: 0.5419
Epoch 9/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7415 - loss: 0.5323
Epoch 10/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7457 - loss: 0.5243
Epoch 11/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7483 - loss: 0.5176
Epoch 12/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/ste

Applying Feature Scaling

In [47]:
# Create scaler
scaler = StandardScaler()

# Fit on training data and transform
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data using the same scaler
X_test_scaled = scaler.transform(X_test)

print("Scaled training data shape:", X_train_scaled.shape)
print("Scaled testing data shape:", X_test_scaled.shape)

print("\nMean of first 5 scaled features:")
print(np.mean(X_train_scaled, axis=0)[:5])

print("\nStandard deviation of first 5 scaled features:")
print(np.std(X_train_scaled, axis=0)[:5])

Scaled training data shape: (7624, 55)
Scaled testing data shape: (1906, 55)

Mean of first 5 scaled features:
[-3.1772466e-08  1.6011320e-08 -5.5038916e-09 -8.7561913e-09
 -1.0507430e-08]

Standard deviation of first 5 scaled features:
[0.99999994 1.         1.         0.99999994 1.        ]


Training the same neural network with feature scaling

In [48]:
# Reset seed for a reproducible initialization
tf.keras.utils.set_random_seed(42)

# Build the same neural network
model_scaled = build_model(X_train_scaled.shape[1])

# Train for 20 epochs
history_scaled = model_scaled.fit(
    X_train_scaled,
    y_train,
    epochs=20,
    batch_size=32,
    verbose=1
)

# Evaluate on the test set
loss_scaled, accuracy_scaled = model_scaled.evaluate(
    X_test_scaled,
    y_test,
    verbose=0
)

print(f"Test Loss with Scaling: {loss_scaled:.4f}")
print(f"Test Accuracy with Scaling: {accuracy_scaled * 100:.2f}%")

Epoch 1/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.5698 - loss: 0.7048
Epoch 2/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7577 - loss: 0.4964
Epoch 3/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7669 - loss: 0.4747
Epoch 4/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7726 - loss: 0.4687
Epoch 5/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7728 - loss: 0.4659
Epoch 6/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7727 - loss: 0.4642
Epoch 7/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7735 - loss: 0.4628
Epoch 8/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7737 - loss: 0.4617
Epoch 9/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7744 - loss: 0.4607
Epoch 10/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7753 - loss: 0.4598
Epoch 11/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7766 - loss: 0.4591
Epoch 12/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

Comparing both accuracies

In [49]:
results = pd.DataFrame({
    "Model": [
        "Without Feature Scaling",
        "With Feature Scaling"
    ],
    "Test Accuracy (%)": [
        accuracy_unscaled * 100,
        accuracy_scaled * 100
    ],
    "Test Loss": [
        loss_unscaled,
        loss_scaled
    ]
})

display(results.round(4))

difference = (accuracy_scaled - accuracy_unscaled) * 100

print("\nComparison:")
if difference > 0:
    print(
        f"Feature scaling improved test accuracy by "
        f"{difference:.2f} percentage points."
    )
elif difference < 0:
    print(
        f"Feature scaling reduced test accuracy by "
        f"{abs(difference):.2f} percentage points."
    )
else:
    print("Feature scaling did not change the test accuracy.")

,Model,Test Accuracy (%),Test Loss
0,Without Feature Scaling,76.6002,0.4803
1,With Feature Scaling,77.4921,0.4624



Comparison:
Feature scaling improved test accuracy by 0.89 percentage points.


Train the network with Sigmoid, Tanh, and ReLU

In [50]:
# Function to build the same neural network
def build_activation_model(activation_function, input_dim):
    model = keras.Sequential([
        keras.Input(shape=(input_dim,)),
        keras.layers.Dense(8, activation=activation_function),
        keras.layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

Train with Sigmoid activation

In [51]:
tf.keras.utils.set_random_seed(42)

model_sigmoid = build_activation_model(
    "sigmoid",
    X_train_scaled.shape[1]
)

history_sigmoid = model_sigmoid.fit(
    X_train_scaled,
    y_train,
    epochs=20,
    batch_size=32,
    verbose=1
)

loss_sigmoid, acc_sigmoid = model_sigmoid.evaluate(
    X_test_scaled,
    y_test,
    verbose=0
)

print(f"Sigmoid Test Accuracy: {acc_sigmoid * 100:.2f}%")

Epoch 1/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4343 - loss: 0.7763
Epoch 2/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7564 - loss: 0.5574
Epoch 3/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7647 - loss: 0.5041
Epoch 4/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7706 - loss: 0.4827
Epoch 5/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7728 - loss: 0.4744
Epoch 6/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7723 - loss: 0.4703
Epoch 7/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7723 - loss: 0.4679
Epoch 8/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7727 - loss: 0.4665
Epoch 9/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7722 - loss: 0.4657
Epoch 10/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7724 - loss: 0.4651
Epoch 11/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7720 - loss: 0.4646
Epoch 12/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

Train with Tanh activation

In [52]:
tf.keras.utils.set_random_seed(42)

model_tanh = build_activation_model(
    "tanh",
    X_train_scaled.shape[1]
)

history_tanh = model_tanh.fit(
    X_train_scaled,
    y_train,
    epochs=20,
    batch_size=32,
    verbose=1
)

loss_tanh, acc_tanh = model_tanh.evaluate(
    X_test_scaled,
    y_test,
    verbose=0
)

print(f"Tanh Test Accuracy: {acc_tanh * 100:.2f}%")

Epoch 1/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6632 - loss: 0.6113
Epoch 2/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7619 - loss: 0.4953
Epoch 3/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7703 - loss: 0.4738
Epoch 4/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7728 - loss: 0.4690
Epoch 5/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7726 - loss: 0.4667
Epoch 6/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7726 - loss: 0.4651
Epoch 7/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7737 - loss: 0.4637
Epoch 8/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7753 - loss: 0.4624
Epoch 9/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7754 - loss: 0.4612
Epoch 10/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7764 - loss: 0.4602
Epoch 11/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7754 - loss: 0.4593
Epoch 12/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

Train with ReLU activation

In [53]:
tf.keras.utils.set_random_seed(42)

model_relu = build_activation_model(
    "relu",
    X_train_scaled.shape[1]
)

history_relu = model_relu.fit(
    X_train_scaled,
    y_train,
    epochs=20,
    batch_size=32,
    verbose=1
)

loss_relu, acc_relu = model_relu.evaluate(
    X_test_scaled,
    y_test,
    verbose=0
)

print(f"ReLU Test Accuracy: {acc_relu * 100:.2f}%")

Epoch 1/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.5698 - loss: 0.7048
Epoch 2/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7577 - loss: 0.4964
Epoch 3/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7669 - loss: 0.4747
Epoch 4/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7726 - loss: 0.4687
Epoch 5/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7728 - loss: 0.4659
Epoch 6/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7727 - loss: 0.4642
Epoch 7/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7735 - loss: 0.4628
Epoch 8/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7737 - loss: 0.4617
Epoch 9/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7744 - loss: 0.4607
Epoch 10/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7753 - loss: 0.4598
Epoch 11/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7766 - loss: 0.4591
Epoch 12/20
239/239 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step

Building a comparison table

In [54]:
results = pd.DataFrame({
    "Activation Function": ["Sigmoid", "Tanh", "ReLU"],
    "Test Accuracy (%)": [
        acc_sigmoid * 100,
        acc_tanh * 100,
        acc_relu * 100
    ]
})

results = results.sort_values(
    by="Test Accuracy (%)",
    ascending=False
).reset_index(drop=True)

display(results.round(2))

best_activation = results.iloc[0]["Activation Function"]
best_accuracy = results.iloc[0]["Test Accuracy (%)"]

print(f"Best activation function: {best_activation}")
print(f"Best test accuracy: {best_accuracy:.2f}%")

,Activation Function,Test Accuracy (%)
0,Sigmoid,77.54
1,ReLU,77.49
2,Tanh,76.97


Best activation function: Sigmoid
Best test accuracy: 77.54%
